# Exploring LLM evaluation

This notebook makes each stage of the local LLM evaluation pipeline visible, so the final metrics can be inspected rather than taken on trust.

## Stage 1: Load the dataset

Evaluation starts with test cases, not model calls. Loading and previewing cases first catches malformed prompts before expensive inference.

In [ ]:
from src.dataset import load_dataset, preview_dataset

dataset = load_dataset("datasets/factual_qa.json")
preview_dataset(dataset, n=3)

## Stage 2: Run the model

The runner isolates local Ollama communication and records latency for every case, which helps explain differences between machines and prompts.

In [ ]:
from config import OLLAMA_MODEL
from src.runner import run_batch

# Run this cell after Ollama is serving qwen2.5:7b locally.
results_df = run_batch(dataset, model=OLLAMA_MODEL)
results_df.head()

## Stage 3: Score the outputs

Separate scorers reveal whether an answer is relevant, factually consistent, or safe. One quality number would hide those differences.

In [ ]:
from src.scorers.hallucination import score_hallucination_batch
from src.scorers.relevance import score_relevance_batch
from src.scorers.toxicity import score_toxicity_batch

scored_df = score_relevance_batch(results_df)
scored_df = score_hallucination_batch(scored_df)
scored_df = score_toxicity_batch(scored_df)
scored_df.head()

## Stage 4: Aggregate the results

Aggregation turns row-level judgments into a scoreboard and highlights weak cases. Worst examples are often more useful for debugging than averages.

In [ ]:
from src.aggregator import (
    compute_dimension_summary,
    compute_overall_pass_rate,
    flag_worst_cases,
)

summary = compute_dimension_summary(scored_df)
summary["overall_pass_rate"] = compute_overall_pass_rate(scored_df)
worst_cases = flag_worst_cases(scored_df, n=5)
summary, worst_cases

## Stage 5: Generate reports

JSON preserves structured data for automation, while the standalone HTML report makes pass and fail patterns easy to scan.

In [ ]:
from src.reporting import generate_html_report, generate_json_report

generate_json_report(summary, scored_df, "results/report.json")
generate_html_report(summary, scored_df, "results/report.html")

## What I Learned

1. Evaluation needs explicit dimensions because relevance, factuality, and safety can fail independently.
2. Rubrics make an LLM judge more consistent than an unconstrained quality request.
3. Structured JSON is useful only when parsing failures are visible and handled.
4. Worst-case examples reveal actionable behavior that averages can hide.
5. The same model as subject and judge is convenient, but independent judges reduce shared blind spots.